# Chapter 10 — The Notebook Is Not the Program You See

**Book alignment:** Debugging AI From First Principles, Chapter 10

**Question this notebook isolates:** A notebook passes interactively and dies on **Restart &
Run All** at cell 14 with `NameError: region_map`. Does reading the `execution_count` gutter
(not the screen) reconstruct the program the kernel actually ran — and does the *location*
of the first failure under a clean run separate **H1** (stale residue from an edited-away
cell) from **H2** (a real top-to-bottom order dependency)?

This models a `.ipynb` as a list of cell dicts, exactly as the JSON on disk stores it.

In [ ]:
# a notebook, as stored: source + the execution_count the kernel stamped on each cell.
# gutter reads 1, 2, 12, 14, 13, 15  -> executed OUT OF ORDER.
NB = [
    {"src": "df = load('sales_v3.parquet')",       "exec": 1,  "defines": ["df"]},
    {"src": "df = clean(df, dropna=True)",          "exec": 2,  "defines": ["df"]},
    {"src": "df = df.assign(q=df.month // 3)",      "exec": 12, "defines": ["df"]},   # EDITED: used to define region_map
    {"src": "summary = summarize(df, region_map)",  "exec": 14, "uses": ["df", "region_map"], "defines": ["summary"]},
    {"src": "chart = plot(summary)",                "exec": 13, "uses": ["summary"], "defines": ["chart"]},
    {"src": "export(summary)",                      "exec": 15, "uses": ["summary"]},
]
# the warm interactive kernel still holds `region_map` from an earlier execution of cell 2
# (index 2) whose text has since been edited away. No current cell defines it.
STALE_BINDINGS = {"region_map"}

## 1. Read the gutter, not the screen — reconstruct the executed program

In [ ]:
by_exec = sorted(range(len(NB)), key=lambda i: NB[i]["exec"])
print("DISPLAY order  :", [i for i in range(len(NB))])
print("EXECUTED order :", by_exec)
non_monotonic = [NB[i]["exec"] for i in range(len(NB))]
assert non_monotonic != sorted(non_monotonic)          # a MEASUREMENT of out-of-order execution
print("\nexecution_count is non-monotonic top-to-bottom -> the screen is not the program that ran")

## 2. Simulate Restart & Run All — a clean kernel, top-to-bottom, no history

In [ ]:
def run_all(nb, *, warm=False):
    defined = set(STALE_BINDINGS) if warm else set()
    for pos, cell in enumerate(nb):                       # top-to-bottom, as Run All executes
        missing = set(cell.get("uses", [])) - defined
        if missing:
            return pos, f"NameError: {sorted(missing)[0]}"
        defined.update(cell.get("defines", []))
    return None, "clean green run"

warm_pos, warm_res = run_all(NB, warm=True)
cold_pos, cold_res = run_all(NB, warm=False)
print(f"interactive (warm kernel): cell {warm_pos} -> {warm_res}")
print(f"Restart & Run All (cold) : cell {cold_pos} -> {cold_res}")
assert warm_res == "clean green run"           # the warm kernel's stale binding hides the gap
assert cold_res.startswith("NameError")

## 3. The location of the first cold failure separates H1 from H2

In [ ]:
# H1 (stale residue): cold run fails at the SAME cell the interactive suspicion pointed at,
#                     because a since-deleted/edited cell used to define the name.
# H2 (order dependence): cold run fails at an EARLIER cell - the top-to-bottom order itself
#                        violates a real dependency.
suspected_cell = 3                      # 'summary = summarize(df, region_map)'
if cold_pos == suspected_cell:
    verdict = "H1 stale residue"
elif cold_pos < suspected_cell:
    verdict = "H2 order dependence"
else:
    verdict = "downstream symptom"
print(f"cold first-failure at cell {cold_pos}; suspected cell {suspected_cell} -> {verdict}")
assert verdict == "H1 stale residue"
print("fix: restore the definition in VISIBLE text (or reorder for H2); then Restart & Run All must be green")
print("a notebook fix verified only by single-cell re-run is unverified")

## What we earned

A notebook is two programs — the displayed document and the executed sequence — and the
kernel only ran the second. The `execution_count` gutter (`1..11, 12, 15, 14, 13`) is a
measurement of out-of-order execution; a clean Restart & Run All is the only admissible
reproduction. Where that clean run *first* fails separates stale residue from a genuine
order dependency. Empirically this displayed-vs-executed gap is the single largest reason a
saved notebook does not reproduce (Pimentel et al.: ~4% reproduce; Samuel & Mietchen: ~8.5%).

**Notebook 11 / Chapter 11** stays in the notebook and hunts the variable that exists in
the kernel but in no visible cell — hidden state.